# Earth Engine local pixel-quality feasibility

Check actual Earth Engine assets and valid/clear mask coverage around the two flagged building outlines. This is a quality-screening notebook, not a flood detector. It preserves the distinction between cloudy/missing observations and dry locations.

Run after notebooks 02 and 03. Cached query responses can be summarized without authentication. Enable the remote cell only to obtain a new raw snapshot after interactive Earth Engine authentication. No large imagery exports are launched. Rerunning replaces derived QA tables and records a new processing run.

In [ ]:
from pathlib import Path
import json

# In Colab, clone the repository and change into it before running this cell.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "config/ethekwini.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the africa-map checkout or one of its subdirectories.")
CONFIG = json.loads((ROOT / "config/ethekwini.yaml").read_text())
print("Project:", CONFIG["project_id"])


In [ ]:
RUN_REMOTE_QUERIES = False  # Enable only when obtaining a new registered raw snapshot.
if RUN_REMOTE_QUERIES:
    assert CONFIG['pixel_quality']['optical_clear_scl_classes'] == [4, 5, 6], 'Review the mask implementation before changing classes.'
    from pathlib import Path
    from datetime import datetime,timezone,timedelta
    import json
    import ee
    import geopandas as gpd
    root=ROOT;cfg=CONFIG
    ee.Initialize(project=cfg['availability']['earth_engine_project'])
    p=root/cfg['pixel_quality']['source_directory'];p.mkdir(parents=True,exist_ok=True)
    footprints=gpd.read_file(root/'data/processed/ethekwini/flagged_site_review.gpkg',layer='osm_outlines').to_crs(4326)
    site_geometries={row.institution_id:ee.Geometry(row.geometry.__geo_interface__,geodesic=False).buffer(cfg['pixel_quality']['context_buffer_m'],1) for row in footprints.itertuples()}
    reference_region=ee.Geometry(footprints.geometry.union_all().__geo_interface__,geodesic=False).bounds()
    requests=cfg['pixel_quality']['requests']
    manifest=json.loads((p/'asset_manifest.json').read_text()) if (p/'asset_manifest.json').exists() else []
    results=json.loads((p/'site_quality_results.json').read_text()) if (p/'site_quality_results.json').exists() else []
    for sensor,date in requests:
     if sum(r['sensor']==sensor and r['query_date']==date for r in results)==len(site_geometries):
      print('Reusing completed quality checks:',sensor,date,flush=True);continue
     end=(datetime.fromisoformat(date)+timedelta(days=1)).date().isoformat()
     collection='COPERNICUS/S1_GRD' if sensor=='S1' else 'COPERNICUS/S2_SR_HARMONIZED'
     images=ee.ImageCollection(collection).filterDate(date,end).filterBounds(reference_region)
     if sensor=='S1':
      images=images.filter(ee.Filter.eq('relativeOrbitNumber_start',43)).filter(ee.Filter.eq('orbitProperties_pass','ASCENDING')).filter(ee.Filter.eq('instrumentMode','IW')).filter(ee.Filter.listContains('transmitterReceiverPolarisation','VV')).filter(ee.Filter.listContains('transmitterReceiverPolarisation','VH'))
      props=['system:index','system:time_start','relativeOrbitNumber_start','orbitProperties_pass','instrumentMode','transmitterReceiverPolarisation','resolution_meters']
     else:
      images=images.filter(ee.Filter.eq('MGRS_TILE','36JTM'))
      props=['system:index','system:time_start','MGRS_TILE','CLOUDY_PIXEL_PERCENTAGE','PROCESSING_BASELINE','PRODUCT_ID']
     metadata=images.map(lambda image:ee.Feature(None,image.toDictionary(props)).set("image_asset_id",ee.String(collection+"/").cat(ee.String(image.get("system:index"))))).getInfo()['features']
     print(sensor,date,'matching assets',len(metadata),flush=True)
     manifest.extend({'sensor':sensor,'query_date':date,'collection':collection,**f['properties']} for f in metadata)
     for info in metadata:
      asset=info['properties']['image_asset_id'];image=ee.Image(asset);scale=cfg['pixel_quality']['sar_scale_m'] if sensor=='S1' else cfg['pixel_quality']['optical_scale_m']
      for site_id,geometry in site_geometries.items():
       kwargs={'geometry':geometry,'scale':scale,'crs':'EPSG:32736','maxPixels':100000,'tileScale':1}
       if sensor=='S1':
        valid=image.select(['VV','VH']).mask().reduce(ee.Reducer.min()).unmask(0,False).rename('valid')
        stack=ee.Image.constant(1).rename('total').addBands(valid)
        metrics=stack.reduceRegion(ee.Reducer.sum(),**kwargs).getInfo()
       else:
        scl=image.select('SCL');spectral_valid=image.select(['B3','B8','B11']).mask().reduce(ee.Reducer.min()).unmask(0,False).rename('spectral_valid')
        scl_valid=scl.mask().unmask(0,False).rename('scl_valid')
        clear=scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).And(spectral_valid).unmask(0,False).rename('conservative_clear')
        stack=ee.Image.constant(1).rename('total').addBands([spectral_valid,scl_valid,clear])
        metrics=stack.reduceRegion(ee.Reducer.sum(),**kwargs).getInfo()
        metrics['scl_histogram']=scl.reduceRegion(ee.Reducer.frequencyHistogram(),**kwargs).getInfo().get('SCL',{})
       record={'site_id':site_id,'sensor':sensor,'query_date':date,'earth_engine_asset_id':asset,'analysis_crs':'EPSG:32736','scale_m':scale,'context_buffer_m':cfg['pixel_quality']['context_buffer_m'],'metrics':metrics,'queried_at_utc':datetime.now(timezone.utc).isoformat()}
       results.append(record);print(site_id,metrics,flush=True)
      # Preserve partial progress after each small image query.
      (p/'asset_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
      (p/'site_quality_results.json').write_text(json.dumps(results,indent=2)+'\n')
else:
    print("Using retained Earth Engine query responses; no remote calls.")


In [ ]:
from datetime import datetime, timezone
import csv
import hashlib
import platform
import shutil
import subprocess
import geopandas as gpd
import pandas as pd
import ee
from heritage_flood.sources import pixel_quality_summary

started = datetime.now(timezone.utc)
run_id = "pixel_quality_" + started.strftime("%Y%m%dT%H%M%S%fZ")
raw = ROOT / CONFIG["pixel_quality"]["source_directory"]
asset_path = raw / "asset_manifest.json"
result_path = raw / "site_quality_results.json"
if not asset_path.exists() or not result_path.exists():
    raise RuntimeError("Raw quality results missing. Enable the remote-query cell after interactive authentication, then rerun.")
assets = json.loads(asset_path.read_text())
records = json.loads(result_path.read_text())
asset_index = {r["image_asset_id"]: r for r in assets}
assert len(asset_index) == len(assets), "Review duplicate query responses"
expected = {(sensor, date, f"osm:way/{way}") for sensor, date in CONFIG["pixel_quality"]["requests"] for way in CONFIG["site_review"]["way_ids"]}
actual = {(r["sensor"], r["query_date"], r["site_id"]) for r in records}
assert actual == expected and len(records) == len(actual)
rows = []
for record in records:
    assert record["analysis_crs"] == CONFIG["analysis_crs"]
    assert record["context_buffer_m"] == CONFIG["pixel_quality"]["context_buffer_m"]
    expected_scale = CONFIG["pixel_quality"]["sar_scale_m"] if record["sensor"] == "S1" else CONFIG["pixel_quality"]["optical_scale_m"]
    assert record["scale_m"] == expected_scale
    asset = asset_index[record["earth_engine_asset_id"]]
    metrics = record["metrics"]
    rows.append({
        "institution_id": record["site_id"], "sensor": record["sensor"], "query_date": record["query_date"],
        "earth_engine_asset_id": record["earth_engine_asset_id"],
        "acquired_at_utc": datetime.fromtimestamp(asset["system:time_start"]/1000, timezone.utc).isoformat(),
        "analysis_crs": record["analysis_crs"], "scale_m": record["scale_m"], "context_buffer_m": record["context_buffer_m"],
        "total_sampling_weight": metrics["total"],
        "valid_sampling_weight": metrics.get("valid") if record["sensor"] == "S1" else metrics.get("spectral_valid"),
        "conservative_clear_sampling_weight": metrics.get("conservative_clear"),
        "scl_histogram_json": json.dumps(metrics.get("scl_histogram"), sort_keys=True),
        "queried_at_utc": record["queried_at_utc"], **pixel_quality_summary(record), "run_id": run_id,
    })
quality = pd.DataFrame(rows).sort_values(["institution_id", "sensor", "query_date"])
output = ROOT / CONFIG["output_directory"]
quality.to_csv(output / "earth_engine_site_quality.csv", index=False)
# Keep separate asset matches rather than changing the historic CDSE manifest.
cdse = pd.read_csv(output / "sentinel_scene_manifest.csv")
matched = []
for asset in assets:
    sensor = asset["sensor"]
    if sensor == "S1":
        key = "_".join(asset["image_asset_id"].split("/")[-1].split("_")[:7])
        candidates = cdse.loc[cdse.catalogue_collection.eq("sentinel-1-grd") & cdse.catalogue_id.map(lambda value: "_".join(value.split("_")[:7]) == key)]
        assert asset["relativeOrbitNumber_start"] == 43 and asset["orbitProperties_pass"] == "ASCENDING"
        method = "same satellite/acquisition start-end/orbit prefix; processing suffix differs"
    else:
        parts = asset["PRODUCT_ID"].split("_")
        candidates = cdse.loc[cdse.catalogue_collection.eq("sentinel-2-l2a") & cdse.catalogue_id.map(lambda value: value.split("_")[0] == parts[0] and value.split("_")[2] == parts[2] and value.split("_")[4:6] == parts[4:6])]
        method = "same platform, sensing identifier, orbit and tile; processing baseline/version differs"
    assert len(candidates) == 1, "Review ambiguous acquisition matching"
    matched.append({"earth_engine_asset_id": asset["image_asset_id"], "catalogue_id": candidates.iloc[0].catalogue_id,
        "query_date": asset["query_date"], "sensor": sensor, "match_method": method,
        "ee_processing_baseline": asset.get("PROCESSING_BASELINE"),
        "cdse_processing_version": candidates.iloc[0].processing_version, "run_id": run_id})
pd.DataFrame(matched).to_csv(output / "earth_engine_asset_matches.csv", index=False)
# Display the actual inspection areas in QGIS, explicitly as QA contexts.
footprints_path = output / "flagged_site_review.gpkg"
contexts = gpd.read_file(footprints_path, layer="osm_outlines").to_crs(CONFIG["analysis_crs"])
contexts.geometry = contexts.geometry.buffer(CONFIG["pixel_quality"]["context_buffer_m"])
contexts = contexts[["institution_id", "geometry"]]
contexts["context_buffer_m"] = CONFIG["pixel_quality"]["context_buffer_m"]
contexts["run_id"] = run_id
contexts.to_file(output / "earth_engine_quality_contexts.gpkg", layer="qa_contexts", driver="GPKG")
summary = {
    "run_id": run_id, "earth_engine_project": CONFIG["availability"]["earth_engine_project"],
    "earth_engine_access": "Initialized successfully; remote quality queries completed", "asset_count": len(assets),
    "site_scene_checks": len(records), "context_definition": "Current OSM building outline buffered by 100 m geodesically in Earth Engine; maxError 1 m",
    "mask_method": "S1 VV/VH combined mask; S2 B3/B8/B11 combined spectral mask and conservative SCL classes 4/5/6; QA60 not used",
    "grid_crs": CONFIG["analysis_crs"], "sar_scale_m": CONFIG["pixel_quality"]["sar_scale_m"], "optical_scale_m": CONFIG["pixel_quality"]["optical_scale_m"],
    "count_interpretation": "Weighted grid-pixel sums; boundary samples are fractional, not integer independent observations",
    "optical_fully_excluded_dates": sorted(quality.loc[quality.sensor.eq("S2") & quality.conservative_clear_fraction.eq(0), "query_date"].unique().tolist()),
    "optical_fully_retained_dates": sorted(quality.loc[quality.sensor.eq("S2") & quality.conservative_clear_fraction.eq(1), "query_date"].unique().tolist()),
    "sar_valid_fraction_min": float(quality.loc[quality.sensor.eq("S1"), "valid_fraction"].min()),
    "conclusion": "No conservative clear samples for 13/18/23 April in either tested context; 28 April is first retained post-event optical date among tested scenes. SAR mask coverage is complete but flood/terrain/speckle validity unassessed.",
    "flood_detection_performed": False, "large_exports_launched": False,
    "limitations": "Local mask screening only; not a flood classification, damage estimate or peak reconstruction. SCL can misclassify; mask validity does not establish SAR suitability. Context summaries are not building-only metrics. Current OSM geometry postdates event for museum. Other optical tiles were not tested.",
}
summary_path = ROOT / "metadata/earth_engine_quality_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n")
print(quality[["institution_id", "sensor", "query_date", "valid_fraction", "conservative_clear_fraction", "quality_status"]].to_string(index=False))


In [ ]:
run_directory = ROOT / "metadata/runs" / run_id
run_directory.mkdir(parents=True, exist_ok=False)
(run_directory / "config.json").write_text(json.dumps(CONFIG, indent=2) + "\n")
env = {"python": platform.python_version(), "earthengine_api": ee.__version__, "geopandas": gpd.__version__, "pandas": pd.__version__}
(run_directory / "environment.json").write_text(json.dumps(env, indent=2) + "\n")
inputs = [asset_path, result_path, footprints_path, output / "sentinel_scene_manifest.csv"]
(run_directory / "input_checksums.json").write_text(json.dumps({str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in inputs}, indent=2) + "\n")
code = [ROOT / "src/heritage_flood/sources.py", ROOT / "notebooks/04_satellite_case_study.ipynb"]
(run_directory / "code_checksums.json").write_text(json.dumps({str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in code}, indent=2) + "\n")
# Preserve exact executed code alongside hashes while the working tree is dirty.
for path in code:
    shutil.copyfile(path, run_directory / path.name)
source_manifest = ROOT / "metadata/earth_engine_quality_source_checksums.json"
source_manifest.write_text(json.dumps({str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in [asset_path, result_path]}, indent=2) + "\n")
outputs = [output / n for n in ["earth_engine_site_quality.csv", "earth_engine_asset_matches.csv", "earth_engine_quality_contexts.gpkg"]] + [summary_path, source_manifest]
(run_directory / "checksums.json").write_text(json.dumps({str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in outputs}, indent=2) + "\n")
dataset_id = "earth_engine_quality_snapshot_20261020"
source_row = {"dataset_id": dataset_id, "title": "Earth Engine candidate-scene asset metadata and local mask summaries",
    "provider": "Google Earth Engine; EU/ESA/Copernicus source imagery", "version": "Query snapshot 2026-10-20; exact EE asset IDs retained",
    "source_url_or_asset_id": "COPERNICUS/S1_GRD;COPERNICUS/S2_SR_HARMONIZED", "status": "accessed",
    "accessed_at_utc": min(r["queried_at_utc"] for r in records), "observation_start": "2022-03-29", "observation_end": "2022-04-28",
    "geographic_coverage": "100 m buffered OSM outlines for two flagged eThekwini sites", "resolution_or_scale": "10 m SAR / 20 m optical QA sampling grids; not damage-resolution claims",
    "crs": CONFIG["analysis_crs"], "license": "Copernicus Sentinel terms; Earth Engine collection references retained",
    "redistribution_conditions": "Small derived QA metrics retained; no imagery export or public publication",
    "limitations": summary["limitations"] + " Register sha256 hashes a raw-response checksum manifest.",
    "storage_location": str(source_manifest.relative_to(ROOT)), "sha256": hashlib.sha256(source_manifest.read_bytes()).hexdigest()}
run_row = {"run_id": run_id, "started_at_utc": started.isoformat(),
    "input_dataset_ids": ";".join([dataset_id, "osm_flagged_sites_snapshot_20261020", "cdse_sentinel_catalogue_snapshot_20261020"]),
    "scene_manifest_path": str((output / "earth_engine_asset_matches.csv").relative_to(ROOT)),
    "code_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
    "working_tree_dirty": str(bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT, text=True).strip())).lower(),
    "config_snapshot_path": str((run_directory / "config.json").relative_to(ROOT)), "environment_path": str((run_directory / "environment.json").relative_to(ROOT)),
    "method": "Authorized Earth Engine masks on 100 m building contexts; SCL 4/5/6 conservative clear rule; weighted sums at explicit CRS/scales; actual EE acquisition matching",
    "output_locations": ";".join(str(p.relative_to(ROOT)) for p in outputs), "output_checksums_path": str((run_directory / "checksums.json").relative_to(ROOT)),
    "validation_status": "partial", "notes": "Asset queries and mask summaries completed. Clear samples are candidates only; no SAR artifacts reviewed, flood masks or damage inference. Exact notebook/module snapshots retained. See research/earth_engine_pixel_quality.md."}
for filename, key, row in [("datasets.csv", "dataset_id", source_row), ("processing_runs.csv", "run_id", run_row)]:
    path = ROOT / "metadata" / filename
    with path.open(newline="") as handle:
        reader = csv.DictReader(handle); fields, rows = reader.fieldnames, list(reader)
    previous = next((r for r in rows if r[key] == row[key]), None)
    if filename == "datasets.csv" and previous and previous["sha256"] != row["sha256"]:
        raise ValueError("Raw query snapshot changed; assign a new dataset ID")
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, lineterminator="\n"); writer.writeheader(); writer.writerows([r for r in rows if r[key] != row[key]] + [row])
print("Registered quality run:", run_id)


## Interpretation

Read `research/earth_engine_pixel_quality.md`. Counts are weighted sampling-grid sums within a 100 m context around the OSM outline, not building-only observations. Conservative SCL classes 4/5/6 identify candidate clear samples, not independently validated visibility. QA60 is unavailable for these April 2022 acquisitions and is not used.

A valid SAR mask does not establish suitability in urban terrain: review speckle, layover/shadow, incidence and comparison windows before any water inference. The April 5/17 pair misses peak timing; dry-looking later imagery cannot rule out earlier flooding. Optical April 13/18/23 contexts have no samples retained by the conservative mask. Among tested post-event dates, April 28 is the first retained candidate. No flood or damage classification is implemented.